# 06. LCEL 체인에 메모리 붙이기 → LCEL 체인을 LangGraph 노드로 감싸기

**legacy 방식**
```python
runnable = RunnablePassthrough.assign(
    chat_history=RunnableLambda(memory.load_memory_variables) | itemgetter("chat_history")
)
chain = runnable | prompt | model
response = chain.invoke({"input": "..."})
memory.save_context({"human": "..."}, {"ai": response.content})   # ← 저장을 직접 해야 함
```

**LangGraph 방식**
* `prompt | model` 체인은 **그대로 재사용** (변경 없음)
* 불러오기 = `state["messages"]`, 저장 = 노드가 반환한 메시지 → checkpointer 가 자동 처리

> legacy 노트북에서 난 `ValueError: variable chat_history should be a list of base messages, got {'chat_history': []}` 는 `| itemgetter("chat_history")` 가 빠진 `runnable` 로 체인을 만들어서 생긴 오류입니다. `load_memory_variables` 는 `{"chat_history": [...]}` **딕셔너리**를 돌려주므로 리스트만 꺼내야 합니다. LangGraph 에서는 이런 연결 코드가 필요 없습니다.

In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## 1. 프롬프트와 체인 (legacy 와 동일)

In [2]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a helpful chatbot"),
        MessagesPlaceholder(variable_name="chat_history"),
        ("human", "{input}"),
    ]
)
chain = prompt | llm

## 2. 체인을 노드로 감싸기

In [3]:
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import InMemorySaver


def call_chain(state: MessagesState):
    *chat_history, last = state["messages"]          # 이전 대화 / 이번 입력 분리
    ai_message = chain.invoke({"chat_history": chat_history, "input": last.content})
    return {"messages": [ai_message]}                # 반환 = 자동 저장 (save_context 불필요)


graph = (
    StateGraph(MessagesState)
    .add_node("chain", call_chain)
    .add_edge(START, "chain")
    .compile(checkpointer=InMemorySaver())
)
config = {"configurable": {"thread_id": "lcel-1"}}

In [4]:
response = graph.invoke({"messages": [HumanMessage("만나서 반갑습니다. 제 이름은 테디입니다.")]}, config)
print(response["messages"][-1].content)

안녕하세요, 테디님! 만나서 반갑습니다. 어떻게 도와드릴까요?


In [5]:
response = graph.invoke({"messages": [HumanMessage("제 이름이 무엇이었는지 기억하세요?")]}, config)
print(response["messages"][-1].content)

네, 테디님이라고 하셨습니다. 다른 질문이나 이야기하고 싶은 것이 있나요?


In [6]:
# legacy: memory.load_memory_variables({})
graph.get_state(config).values["messages"]

[HumanMessage(content='만나서 반갑습니다. 제 이름은 테디입니다.', additional_kwargs={}, response_metadata={}, id='6b41f194-6c47-4979-a305-739f02639ce4'),
 AIMessage(content='안녕하세요, 테디님! 만나서 반갑습니다. 어떻게 도와드릴까요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 30, 'total_tokens': 50, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_0bf9805b5a', 'id': 'chatcmpl-EPMKtJAi2SbW9cpBy3MGRy9mIqksk', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0b331-9a25-7030-9c5e-f65177df588f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 30, 'output_tokens': 

## 3. 사용자 입력 키를 그대로 쓰고 싶다면
legacy 처럼 `{"input": "..."}` 형태로 호출하고 싶으면 state 에 필드를 추가하면 됩니다. 입력 키(`input`)와 기록(`messages`)을 분리하는 패턴입니다.

In [7]:
from langchain_core.messages import AIMessage


class ChainState(MessagesState):
    input: str


def call_chain_with_input(state: ChainState):
    ai_message = chain.invoke({"chat_history": state["messages"], "input": state["input"]})
    return {"messages": [HumanMessage(state["input"]), ai_message]}   # 이번 턴을 기록에 추가


graph2 = (
    StateGraph(ChainState)
    .add_node("chain", call_chain_with_input)
    .add_edge(START, "chain")
    .compile(checkpointer=InMemorySaver())
)
cfg = {"configurable": {"thread_id": "lcel-2"}}
graph2.invoke({"input": "제 취미는 등산입니다."}, cfg)
print(graph2.invoke({"input": "제 취미가 뭐라고 했죠?"}, cfg)["messages"][-1].content)

당신의 취미는 등산이라고 하셨습니다. 등산에 대해 더 이야기하고 싶으신가요?
